## Function: Setup PostGIS + OSM Workflow 🗄️

In this notebook, you'll learn how to build the `setup_postgis_osm()` function step by step. This workflow sets up a spatial database, downloads OpenStreetMap data, and loads it into PostGIS.

### 🎯 What This Function Does
- Connect to a PostgreSQL server
- Create a new database
- Enable PostGIS extension
- Download OSM data from Geofabrik
- Load OSM data into PostGIS

### 🔧 Function Signature
```python
def setup_postgis_osm(osm_url: str,
    db_name: str = "osm_db",
    user: str = "postgres",
    password: str = "postgres",
    host: str = "localhost",
    port: int = 5432,
    data_dir: Optional[Path] = None):
    """      
    Args:
    osm_url: URL to OSM .pbf file (Geofabrik download)
    db_name: Name of the database to create
    user: PostgreSQL username
    password: PostgreSQL password
    host: Database host
    port: Database port
    data_dir: Directory to store downloaded OSM data

    Returns:
        None
    """
```

### 📍 Where This Function Goes:
**Target File**: `src/setup_postgis_osm.py`  
**Function Name**: `setup_postgis_osm()`  
**Replace**: The placeholder function with your working code

---

### ⚙️ Step 0: Select the Correct Python Kernel

Before running any cells, make sure the notebook is using the correct Python environment.

**Check the kernel in the top-right corner of the notebook.**

The correct Python environment is **python-gis-postgis-development (.venv)**  
It may appear with a Python version, for example:  
**python-gis-postgis-development (Python 3.11.15)**

If the kernel is **python-gis-postgis-development (Python 3.11.15)**, you can start running cells below.

Steps to select the correct kernel:
1. Click on the kernel (top right corner of the notebook) if it is not **python-gis-postgis-development (Python 3.11.15)** or if it says "Select Kernel"
2. Select **python-gis-postgis-development (Python 3.11.15)**
3. If you do not see the kernel in the list, click on "Select Another Kernel..."  
    a. Click on Python Environments...  
    b. Select **python-gis-postgis-development (Python 3.11.15)**

Once the correct kernel is selected, you can start running cells below.

### 📚 Step 1: Import Required Libraries

We will use the following tools:
- `psycopg2`: connect to PostgreSQL
- `requests`: download OSM data
- `subprocess`: run system commands (`osm2pgsql`)
- `os`: manage file paths

**💡 These will be used in our function!**

In [ ]:
import os
import requests
import subprocess
import psycopg2

print("Libraries imported!")

### 🔌 Step 2: Connect to PostgreSQL Server

We will first connect to the default `postgres` database.

**💡 This will be used in our function!**

In [ ]:
# Establish a connection to the postgres database
conn = psycopg2.connect(
    dbname="postgres",
    user="postgres",
    password="postgres",
    host="localhost",
    port=5432
)

# Every SQL statement is executed immediately
conn.autocommit = True

# Open a cursor to perform database operations
cur = conn.cursor()

# Check current database by using PostgreSQL built-in function
cur.execute("SELECT current_database();")

# Use fetchone() to return one row from the result of the query
print("Current database:", cur.fetchone()[0])

print("Connected to PostgreSQL server")

### 🗄️ Step 3: Create a New Database

We will create a new database `osm_db` where OSM data will be stored.

**💡 This will be used in our function!**

In [ ]:
# Check if database osm_db already exists
cur.execute("SELECT 1 FROM pg_database WHERE datname = 'osm_db';")

# fetchone() returns a row if it exists, otherwise None
exists = cur.fetchone()

if not exists:
    cur.execute("CREATE DATABASE osm_db;")
    print("Database 'osm_db' created")
else:
    print("Database 'osm_db' already exists")

# Verify again (explicit confirmation)
cur.execute("SELECT datname FROM pg_database WHERE datname = 'osm_db';")
print("Verified:", cur.fetchone()[0])

# Close connection to 'postgres' before switching databases
cur.close()
conn.close()

print("Closed connection to 'postgres'")

### 🌍 Step 4: Enable PostGIS Extension

PostGIS adds spatial capabilities to PostgreSQL. Without this step, we cannot store or analyze spatial data.

**💡 This will be used in our function!**

In [ ]:
# Establish a connection to the osm_db database
conn = psycopg2.connect(
    dbname="osm_db",
    user="postgres",
    password="postgres",
    host="localhost",
    port=5432
)

# Every SQL statement is executed immediately
conn.autocommit = True

# Open a cursor to perform database operations
cur = conn.cursor()

print("Connected to database:", "osm_db")

# Create extension if it does not exist
cur.execute("CREATE EXTENSION IF NOT EXISTS postgis;")

# Check PostGIS version
cur.execute("SELECT PostGIS_version();")
version = cur.fetchone()
print("PostGIS version:", version)

### ⬇️ Step 5: Download OSM Data from Geofabrik

Geofabrik provides pre-packaged OpenStreetMap extracts as `osm.pbf` files (compressed binary format). We will download Hawaii data as our example.

**💡 This will be used in our function!**

In [ ]:
# URL for OSM data download (Geofabrik)
osm_url = "https://download.geofabrik.de/north-america/us/hawaii-latest.osm.pbf"

# Define local directory to store OSM data
data_path = "../data/osm"

# Create directory if it does not exist
os.makedirs(data_path, exist_ok=True)

# Construct full file path using the filename from the URL
file_path = os.path.join(data_path, osm_url.split("/")[-1])

# Download file only if it does not already exist
if not os.path.exists(file_path):
    print("Downloading OSM data...")
    print("URL:", osm_url)

    # Send HTTP request (stream=True downloads in chunks)
    response = requests.get(osm_url, stream=True, timeout=300)
    # Raise error if download fails
    response.raise_for_status()
    
    # Get total file size (if available) for progress tracking
    file_size = int(response.headers.get("content-length", 0))
    if file_size > 0:
        print(f"File size: {file_size / (1024 * 1024):.1f} MB")

    downloaded = 0
    
    # Open file in binary write mode
    with open(file_path, "wb") as f:
        # Download file in chunks to avoid loading entire file into memory
        for chunk in response.iter_content(chunk_size=8192):
            if chunk:
                f.write(chunk)
                downloaded += len(chunk)
                
                # Display progress as percentage (if file size is known)
                if file_size > 0:
                    progress = downloaded / file_size * 100
                    print(f"\rProgress: {progress:.1f}%", end="", flush=True)

    print("\nDownload complete")
    print("Saved to:", file_path)
else:
    # Skip download if file already exists locally
    print("File already exists:")
    print(file_path)

### 📥 Step 6: Load OSM Data into PostGIS

We use `osm2pgsql` to load OSM `.pbf` data into PostGIS database tables. This command runs in a unix shell.

**Command Explanation:**
- `osm2pgsql`: converts OSM data to SQL
- `-d`: database name
- `-U`: database user
- `--create`: create new tables
- `--slim`: required for larger datasets

**💡 This will be used in our function!**

In [ ]:
# Build osm2pgsql command to load OSM data into PostGIS
cmd = [
    "osm2pgsql",
    "-d", "osm_db",
    "-U", "postgres",
    "-H", "localhost",
    "--create",
    "--slim",
    file_path
]

print("Running osm2pgsql...")
print("Command:", " ".join(cmd))

# Set password so osm2pgsql does not prompt for input
env = os.environ.copy()
env["PGPASSWORD"] = "postgres"

# Run osm2pgsql and display output in the notebook
try:
    subprocess.run(cmd, env=env, check=True)
    print("OSM data loaded into PostGIS successfully")
except subprocess.CalledProcessError as e:
    print("osm2pgsql failed")
    print(e)

### ✅ Step 7: Verify OSM Data Load

We now confirm that the OSM data was successfully loaded into PostGIS.

**What this check does:**
- lists all tables created in the database  
- verifies that key OSM tables exist (`planet_osm_point`, `planet_osm_line`, `planet_osm_polygon`)  
- counts the number of rows in each table  

**💡 If row counts are zero or tables are missing, the data load did not complete correctly**

In [ ]:
# Create an SQL query as a python string literal to list all tables
list_query = """
SELECT table_name 
FROM information_schema.tables
WHERE table_schema = 'public'
ORDER BY table_name;
"""

# Execute the query
cur.execute(list_query)

# use fetchall() to retrieve all rows returned by the SQL query
tables = [t[0] for t in cur.fetchall()]
print("Tables created:", tables)

# Row counts in 3 key tables
for table in ["planet_osm_point", "planet_osm_line", "planet_osm_polygon"]:
    try:
        # Execute count query for each table
        cur.execute(f"SELECT COUNT(*) FROM {table};")
        count = cur.fetchone()[0]
        print(f"{table}: {count} rows")
    except:
        # Handle case where table does not exist
        print(f"{table}: not found")

# Close the database connection
cur.close()
conn.close()

print("Database connection closed")

### 🧩 Step 7: Build the Complete Function

Now we combine all steps into a reusable function.

In [ ]:
def setup_postgis_osm(db_name="osm_db", osm_url=None):
    import os, requests, subprocess, psycopg2

    data_path = "data/osm"
    os.makedirs(data_path, exist_ok=True)

    file_path = os.path.join(data_path, osm_url.split("/")[-1])

    if not os.path.exists(file_path):
        r = requests.get(osm_url, stream=True)
        with open(file_path, "wb") as f:
            for chunk in r.iter_content(1024):
                f.write(chunk)

    conn = psycopg2.connect(dbname="postgres", user="postgres", password="postgres", host="localhost", port=5432)
    conn.autocommit = True
    cur = conn.cursor()
    cur.execute(f"CREATE DATABASE {db_name};")
    cur.close()
    conn.close()

    conn = psycopg2.connect(dbname=db_name, user="postgres", password="postgres", host="localhost", port=5432)
    conn.autocommit = True
    cur = conn.cursor()
    cur.execute("CREATE EXTENSION postgis;")
    cur.close()
    conn.close()

    subprocess.run([
        "osm2pgsql",
        "-d", db_name,
        "-U", "postgres",
        "--create",
        "--slim",
        file_path
    ], check=True)

    return True